In [ ]:
!pip install evaluate bert_score sentence-transformers

In [ ]:
from peft import PromptTuningConfig, get_peft_model, TaskType, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import os
from tqdm import tqdm
import torch
import re
from evaluate import load
import json
from sentence_transformers import SentenceTransformer, util
from datasets import Dataset

In [ ]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        trust_remote_code=True
    )

    return tokenizer, model

In [ ]:
model_name = "google/gemma-3-1b-it"

tokenizer, base_model = load_model(model_name)

Prepare the dataset

In [ ]:
def clean_punctuation(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r"\s'(\w)", r"'\1", text)
    text = re.sub(r"\s([.,!?;:])", r"\1", text)
    text = text.replace(" n't", "n't")
    return text.strip()

In [ ]:
df = pd.read_csv("pie_corpus.csv")

text_columns = ['Literal_Sent', 'Idiomatic_Sent', 'Idiom']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_punctuation)

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df, groups=df['Idiom']))

train_val_df = df.iloc[train_val_idx]
test_df = df.iloc[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=42)
train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['Idiom']))

train_df = train_val_df.iloc[train_idx]
val_df = train_val_df.iloc[val_idx]

print(f"Idioms in Train: {train_df['Idiom'].nunique()}, Lines: {len(train_df)}")
print(f"Idioms in Val: {val_df['Idiom'].nunique()}, Lines: {len(val_df)}")
print(f"Idioms in Test: {test_df['Idiom'].nunique()}, Lines: {len(test_df)}")

In [ ]:
train_dataset_raw = Dataset.from_pandas(train_df)
eval_dataset_raw = Dataset.from_pandas(val_df)

In [ ]:
direction = "to_literal"

In [ ]:
def tokenize_function(examples, direction='to_idiomatic'):
    if direction == 'to_idiomatic':
        prompts = [
            f"Meaning: {s}. Sentence: {l}. Resulting sentence:"
            for s, l in zip(examples["Sense"], examples["Literal_Sent"])
        ]

        targets = [
            " " + t + tokenizer.eos_token
            for t in examples["Idiomatic_Sent"]
        ]

    else:
        prompts = [
            f"Meaning: {s}. Sentence: {l}. Resulting sentence:"
            for s, l in zip(examples["Sense"], examples["Idiomatic_Sent"])
        ]

        targets = [
            " " + t + tokenizer.eos_token
            for t in examples["Literal_Sent"]
        ]

    prompt_tokens = tokenizer(prompts, add_special_tokens=False)
    target_tokens = tokenizer(targets, add_special_tokens=False)

    input_ids = []
    attention_mask = []
    labels = []

    for p, t in zip(prompt_tokens["input_ids"], target_tokens["input_ids"]):
        ids = p + t
        lab = [-100] * len(p) + t

        max_len = 200
        ids = ids[:max_len]
        lab = lab[:max_len]

        pad_len = max_len - len(ids)

        ids += [tokenizer.pad_token_id] * pad_len
        lab += [-100] * pad_len

        mask = [1] * (max_len - pad_len) + [0] * pad_len

        input_ids.append(ids)
        attention_mask.append(mask)
        labels.append(lab)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
old_column_names = train_dataset_raw.column_names

train_dataset = train_dataset_raw.map(
    lambda examples: tokenize_function(examples, direction=direction),
    batched=True,
    remove_columns=old_column_names
)
eval_dataset = eval_dataset_raw.map(
    lambda examples: tokenize_function(examples, direction=direction),
    batched=True,
    remove_columns=old_column_names
)

Prompt Tuning

In [ ]:
clean_model_name = model_name.split('/')[-1]

In [ ]:
def calculate_metrics(csv_path, ref_lookup, lang="en"):
    df = pd.read_csv(csv_path)

    meteor_metric = load("meteor")
    bertscore_metric = load("bertscore")
    bert_score_model = "roberta-large" if lang == "en" else "xlm-roberta-large"

    if lang == "en":
        sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    else:
        sbert_model = SentenceTransformer('sentence-transformers/LaBSE')


    preds = df['generated_text'].fillna("").tolist()
    inputs = df['input_text'].tolist()

    refs = [ref_lookup.get(inp, [df.iloc[i]['reference']]) for i, inp in enumerate(inputs)]

    meteor_res = meteor_metric.compute(predictions=preds, references=refs)

    all_bs_f1 = []
    for p, rs in zip(preds, refs):
        res = bertscore_metric.compute(predictions=[p] * len(rs), references=rs, lang=lang, model_type=bert_score_model)
        all_bs_f1.append(max(res['f1']))

    max_cosines = []
    for pred, rs in zip(preds, refs):
        p_emb = sbert_model.encode([pred], convert_to_tensor=True)
        r_embs = sbert_model.encode(rs, convert_to_tensor=True)
        scores = util.cos_sim(p_emb, r_embs)
        max_cosines.append(scores.max().item())

    return {
        "METEOR": round(meteor_res['meteor'], 4),
        "BERTScore_F1": round(sum(all_bs_f1) / len(all_bs_f1), 4),
        "Cosine_Similarity": round(sum(max_cosines) / len(max_cosines), 4),
    }

In [ ]:
def get_peft_config(direction='to_idiomatic'):
    if direction == 'to_idiomatic':
        prompt_text = "Rewrite the sentence to include an idiom that matches the meaning provided:"
    else:
        prompt_text = "Replace the idiom in the sentence with a literal expression that matches the meaning provided:"

    peft_config = PromptTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=25,
        prompt_tuning_init="TEXT",
        prompt_tuning_init_text=prompt_text,
        tokenizer_name_or_path=model_name,
    )

    return peft_config

In [ ]:
peft_config = get_peft_config(direction=direction)

model = get_peft_model(base_model, peft_config)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./{clean_model_name}/{direction}/checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-2,
    num_train_epochs=3,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=True,
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()

In [ ]:
model.save_pretrained(f"./{clean_model_name}/{direction}")

In [ ]:
model = PeftModel.from_pretrained(base_model, f"./{clean_model_name}/{direction}")

model.to("cuda")
model.eval()

In [ ]:
generated_results = []

for item in tqdm(test_df.to_dict('records')):

    prompt = f"Meaning: {item['Sense']}. Sentence: {item['Idiomatic_Sent']}. Resulting sentence:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    generated_results.append({
        "idiom": item['Idiom'],
        "input_text": prompt,
        "generated_text": result,
        "reference": item['Literal_Sent']
    })

In [ ]:
df_res = pd.DataFrame(generated_results)
df_res.to_csv(f"./{clean_model_name}/{direction}/prompt_tuning_results.csv", index=False)

In [ ]:
metrics = calculate_metrics(f"./{clean_model_name}/{direction}/prompt_tuning_results.csv")

In [ ]:
metrics["direction"] = direction

df_metrics = pd.DataFrame([metrics])

out_path = f"./{clean_model_name}/metrics_log_en.csv"

if os.path.exists(out_path):
    df_metrics.to_csv(out_path, mode="a", header=False, index=False)
else:
    df_metrics.to_csv(out_path, index=False)